WARNING: to use this file please change all filepaths to match your Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Code to randomly populate destination folder with images (IGNORE if your destination folder already has images)

However, all images must have the file format "{image_name}...|{channel_name}.jpg" in order for caption extraction to work.

In [2]:
import os
import random
import shutil

In [3]:
# Set paths (change for your Google Drive)
source_dir = "/content/drive/My Drive/digital/media"
dest_dir = "/content/drive/My Drive/all/media/Coding/Week 6/Images"
n_images_to_copy = 15

def collect_images(dest_dir, n_images_to_copy):
  # Create destination directory if it doesn't exist
  os.makedirs(dest_dir, exist_ok=True)

  folders = ['/occupied', '/deoccupied', '/frontline']  # the categories from which images can be drawn

  all_images = []

  # Collect all image file paths
  for folder in folders:
      for subdir, _, files in os.walk(source_dir+folder+"/random200"):
          for file in files:
              if os.path.splitext(file)[1].lower() in ['.jpg', '.jpeg', '.png']:
                  full_path = os.path.join(subdir, file)
                  parent_folder = os.path.basename(subdir)
                  all_images.append((full_path, parent_folder, file))

  # if there are too few total images
  if len(all_images) < n_images_to_copy:
      raise ValueError(f"Requested {n_images_to_copy} images, but only found {len(all_images)} total.")

  # sample from all images randomly
  sampled = random.sample(all_images, n_images_to_copy)

  # rename and copy images
  for idx, (src_path, subdir_name, img_name) in enumerate(sampled):
      _, ext = os.path.splitext(src_path)
      img_name, _ = os.path.splitext(img_name)
      new_filename = f"{img_name}|{subdir_name}{ext}"   # rename image to include channel name after '|' character
      print(new_filename)
      dst_path = os.path.join(dest_dir, new_filename)
      shutil.copy2(src_path, dst_path)

  print(f"Copied {n_images_to_copy} images to: {dest_dir}")


In [ ]:
# Set paths (change for your Google Drive)
source_dir = "/content/drive/MyDrive/digital_frontline (1)/Visual Media/Images/deoccupied/random200"
dest_dir = "/content/drive/MyDrive/digital_frontline (1)/Visual Media/Coding/Spring 2026/Week 1/Images"

def collect_images(dest_dir):
  # Create destination directory if it doesn't exist
  os.makedirs(dest_dir, exist_ok=True)

  folders = ['beryslavrda', 'countryfreekherson', 'hercon_ru', 'beryslavru', 'DN_Kherson']  # the categories from which images can be drawn

  all_images = []

  # Collect all image file paths
  for folder in folders:
      for subdir, _, files in os.walk(source_dir+'/'+folder):
          for file in files:
              if os.path.splitext(file)[1].lower() in ['.jpg', '.jpeg', '.png']:
                  full_path = os.path.join(subdir, file)
                  parent_folder = os.path.basename(subdir)
                  all_images.append((full_path, parent_folder, file))


  # rename and copy images
  for idx, (src_path, subdir_name, img_name) in enumerate(all_images):
      _, ext = os.path.splitext(src_path)
      img_name, _ = os.path.splitext(img_name)
      new_filename = f"{img_name}|{subdir_name}{ext}"   # rename image to include channel name after '|' character
      print(new_filename)
      dst_path = os.path.join(dest_dir, new_filename)
      shutil.copy2(src_path, dst_path)

  print(f"Copied {len(all_images)} images to: {dest_dir}")


In [ ]:
collect_images(dest_dir)

2062_FUI_AgAD48YxG9UgkUk|beryslavrda.jpg
973_FUI_AgADwsMxG9oRUUo|beryslavrda.jpg
981_FUI_AgAD-MIxG2hNYUo|beryslavrda.jpg
4421_FUI_AgAD1swxG0Z5YUk|countryfreekherson.jpg
5809_FUI_AgADtsAxG1gjuUo|hercon_ru.jpg
586_FUI_AgADrLwxG0momEs|hercon_ru.jpg
1864_FUI_AgADUcQxG7KS4Eg|beryslavru.jpg
7931_FUI_AgADYcIxG20LWUo|beryslavru.jpg
1117_FUI_AgADgb0xGxrYOEg|beryslavru.jpg
15232_FUI_AgADAs8xG5ZPuUk|beryslavru.jpg
307_FUI_AgADBLwxG_Y5QUs|beryslavru.jpg
11636_FUI_AgADiskxG8ZOiEk|beryslavru.jpg
4146_FUI_AgADdrsxG2xVqUo|DN_Kherson.jpg
3043_FUI_AgADuLwxG6EdqUk|DN_Kherson.jpg
842_FUI_AgADIrMxGzD6aUs|DN_Kherson.jpg
Copied 15 images to: /content/drive/MyDrive/digital_frontline (1)/Visual Media/Coding/Spring 2026/Week 1/Images


# Caption Extraction

Note: all image names MUST follow the format "{post_id}...|{channel_name}.jpg"! **This differs from what is directly in the Google Drive!** In the raw names currently in Google Drive, there is no way of directly identifying the channel the image is from.

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
# change to match your own Google Drive
message_source_dir = "/content/drive/MyDrive/digital_frontline/Data/Telegram Messages"
images_dir = "/content/drive/MyDrive/digital_frontline/Visual Media/Coding/Spring 2026/Week 3/sp26 week 3 more images"

In [ ]:
# ADDED CODE: indexing image names to their folder and channel for ease of access

image_name_index = {}

def index_images(data_path):
  if image_name_index != {}:
    return image_name_index
  for root, dirs, files in os.walk(data_path):
    for file_name in files:
      df = pd.read_json(os.path.join(root, file_name))
      for _, row in df.iterrows():
        image_name = row['id']
        channel = row['channel_username']
        image_name_index[image_name] = (channel, root)
  return image_name_index


In [ ]:
# helper function to get channel data json given channel name
def get_channel_data(data_path, channel, folder=None):
    df = None
    if folder:
        file_path = f"{data_path}/{folder}/{channel}.json"
        df = pd.read_json(file_path)
        return df
    else:
        for root, dirs, files in os.walk(data_path):
            for file_name in files:
                if file_name == f"{channel}.json":
                    df = pd.read_json(os.path.join(root, file_name))
                    return df
        print(f"CHANNEL {channel} NOT FOUND")
        return None

# helper function to get caption data
def get_message(channel_df, post_id):
    try:
      message = channel_df.loc[channel_df['id'] == post_id, 'caption'].iloc[0]
      if message and message != 'null':
        return message
      else:
          print(' NO CAPTION FOUND for post', post_id, 'in channel', channel_df.loc[channel_df['id'] == post_id, 'channel_username'].iloc[0])
          return None
    except:
      print(' ', post_id, 'NOT FOUND in channel', channel_df.loc[channel_df['id'] == post_id, 'channel_username'].iloc[0])
      return None

def get_caption_of_image(data_path, image_name, folder=None):
    """
    Function for retrieving the caption (or lack thereof) for an image from Telegram.
    inputs:
        data_path       -   path from current location to directory containing folders (deoccupied, occupied, frontline, etc.)
        image_name      -   name of image as shown in Google Drive
        folder          -   (optional) if known which folder (deoccupied, occupied, frontline, etc.) channel is from

    output:
        (string) content of image caption in original Telegram message, or, if no caption, None
    """
    channel = image_name.split('|')[-1].split('.')[0]
    channel_df = get_channel_data(data_path, channel, folder)
    if channel_df is None:
      return None
    post_id = int(image_name.split('_')[0])
    caption = get_message(channel_df, post_id)
    return caption

def get_captions_of_images_from_list(data_path, images):
    """
    Function for retrieving the captions (or lack thereof) from images from a LIST OF IMAGE NAMES.
    inputs:
        data_path     -   path to directory containing folders of messages (deoccupied, occupied, frontline, etc.)
        image_folder  -   list of image names
        delete        -   whether to delete images with no caption from the directory
        replace       -   whether to replace images with no caption until directory contains all images with captions. only works if delete=True

    output:
        (list) tuples of image name alongside contents of image caption in original Telegram message (string), or, if no caption, None
    """
    captions = []

    for image_name in tqdm(images):
        caption = get_caption_of_image(data_path, image_name)
        captions.append((image_name, caption))
    return captions


def get_captions_of_images_from_folder(data_path, image_folder, delete=True, replace=True):
    """
    Function for retrieving the captions (or lack thereof) from images from a DIRECTORY OF IMAGES.
    inputs:
        data_path     -   path to directory containing folders of messages (deoccupied, occupied, frontline, etc.)
        image_folder  -   path to directory containing images to get captions of
        delete        -   whether to delete images with no caption from the directory
        replace       -   whether to replace images with no caption until directory contains all images with captions. only works if delete=True

    output:
        (list) tuples of image name alongside contents of image caption in original Telegram message (string), or, if no caption, None
    """
    captions = []
    delete_count = 0

    for _, _, files in os.walk(image_folder):
        for image_name in tqdm(files):
          caption = get_caption_of_image(data_path, image_name)
          if not caption and delete:
            file_path = f"{image_folder}/{image_name}"
            if os.path.exists(file_path):
              try:
                  os.remove(file_path)
                  print(f"Image '{image_name}' deleted successfully.")
              except PermissionError:
                  print(f"Permission denied to delete file '{image_name}'.")
              except Exception as e:
                  print(f"An error occurred: {e}")
            else:
                print(f"File '{image_name}' does not exist.")
            delete_count += 1
            continue
          captions.append((image_name, caption))
    if replace:
      if not delete:
        print("You can only replace images without captions if you delete them first. Change delete=True in this function call.")
      else:
        if delete_count > 0:
          print(delete_count,'image(s) without captions deleted. Now replacing:')
          collect_images(image_folder, delete_count)
          print("Performing caption extraction again:")
          captions = get_captions_of_images_from_folder(data_path, image_folder, delete=True, replace=True)
    print("All captions successfully extracted!")
    # sort to ensure order is deterministic and known
    captions.sort(key=lambda x: x[0])
    return captions

In [ ]:
# read documentation about delete and replace parameters (above in function definition) before using!
captions = get_captions_of_images_from_folder(message_source_dir, images_dir, delete=False, replace=False)

 11%|█         | 2/18 [00:02<00:14,  1.11it/s]

 NO CAPTION FOUND for post 8831 in channel jurnko


 28%|██▊       | 5/18 [00:07<00:21,  1.66s/it]

 NO CAPTION FOUND for post 6453 in channel kavun_ks


 56%|█████▌    | 10/18 [00:13<00:09,  1.15s/it]

...

 NO CAPTION FOUND for post 6466 in channel jurnko


 94%|█████████▍| 17/18 [00:19<00:01,  1.04s/it]

 NO CAPTION FOUND for post 7688 in channel kavun_ks


100%|██████████| 18/18 [00:21<00:00,  1.18s/it]

All captions successfully extracted!


In [ ]:
print(captions)

[('12079_FUI_AgADa8UxGymCwUs|kavun_ks.jpg', '⚡️Борис Джонсон заявив, що претендує на посаду генерального секретаря НАТО\n\n➡️ ПІДПИСАТИСЬ НА КАНАЛ\n👉 Надіслати новину'), ('1487_FUI_AgADM7sxG9wQ-Us|kavun_ks.jpg', 'На оккупированном берегу Каховского водохранилища заметили движение вражеской техники.\n\nСо стороны оккупированного Энергодара и близлежащих населенных пунктов отмечено движение вражеской техники, – сообщает глава Никопольского РВА Евгений Евтушенко.\n\nКаховський Кавун 🍉'), ('15941_FUI_AgAEyzEbUBPISQ|kavun_ks.jpg', 'Херсонська область. Інформація щодо ворожих обстрілів за минулу добу 16 квітня.\n\nЗа інформацією Херсонської ОВА, російські окупанти 46 разів обстріляли мирні населені пункти Херсонщини. Випустили 170 снарядів з важкої артилерії, безпілотників та авіації. \n\nЖитлові квартали Херсона армія рф атакувала тричі.\n\nЗа минулу добу постраждалих серед цивільного населення немає.\n\n➡️ ПІДПИСАТИСЬ НА КАНАЛ\n👉 Надіслати новину'), ('18640_FUI_AgADcMcxG12TMEs|kavun_ks.jpg

# Creating Coding Spreadsheets

In [ ]:
!pip install --quiet gspread google-auth googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.13.3 which is incompatible.
mcp 1.26.0 requires httpx>=0.27.1, but you have httpx 0.13.3 which is incompatible.
gradio 5.50.0 requires httpx<1.0,>=0.24.1, but you ha

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
from googletrans import Translator
import requests
# from pydrive.auth import GoogleAuth
# from pydrive.drive import GoogleDrive

# Authenticate
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# this will only populate the captions in the spreadsheet! the other coding parts will have to be copied and pasted
def create_spreadsheet(sheet_folder_id, sheet_name, image_folder_path, image_caption_pairs):
  """
  Function that creates and populates a Google Sheet with captions for images in the folder, in sorted order. Meant to work with the
  Google Apps Script below.

  inputs:
    sheet_folder_id      -  id of folder where target Google Sheet will reside (can be found from folder URL)
    sheet_name           -  desired name for Google Sheet
    image_folder_path    -  path to folder containing images
    image_caption_pairs  -  list of (image_name, caption) pairs, as output from get_captions_of_images_from_folder

  output: void
  """
  # create a new spreadsheet
  sh = gc.create(sheet_name)
  worksheet = sh.sheet1
  file_id = sh.id
  url = f'https://www.googleapis.com/drive/v3/files/{file_id}?addParents={sheet_folder_id}&removeParents=root&fields=id,parents'
  headers = {'Authorization': f'Bearer {creds.token}'}

  res = requests.patch(url, headers=headers)
  if res.status_code == 200:
      print("Moved successfully to the folder.")
  else:
      print("Failed to move:", res.json())

  # share it with your Google account (so you can view it)
  sh.share('ayl27@mit.edu', perm_type='user', role='writer')

  # for translating captions
  translator = Translator()

  rows = []

  # translate captions
  # image name is not used because it will be filled in by the Google Script
  # image/caption pairs are sorted alphabetically to ensure the script and this code have same order of images
  for _, caption in tqdm(image_caption_pairs):
      try:
          translated = translator.translate(caption, dest='en').text
      except:
          translated = 'TRANSLATION ERROR'

      rows.append([caption, translated])

  rows_formula = [[None, None, None, cap, trans] for cap, trans in rows]
  worksheet.update([['Image Name', 'Image URL', 'Image Preview', 'Caption (Original)', 'Translated Caption']] + rows_formula)




In [ ]:
# example
create_spreadsheet('1O63dutFjZFlXnaHv3vyhiu1CARhyoQ6i', 'captions', images_dir, captions)

Moved successfully to the folder.


100%|██████████| 18/18 [00:14<00:00,  1.20it/s]


WARNING: the translation API used here kind of sucks (worse than just plugging the caption into Google Translate)! If a caption makes no sense, consult online Google Translate or even an LLM for hopefully a more coherent translation.

####Then, in the newly created spreadsheet, attach and run this script (NOT in Python, will not run here):

In [ ]:
// populates the sheet with images
function listImagesInFolder() {
  const folderId = '1RGA5DGjORl3XDKcemMreq0b648pra5Lk'; // Replace with ID of folder containing images
  const sheet = SpreadsheetApp.getActiveSpreadsheet().getActiveSheet();
  const folder = DriveApp.getFolderById(folderId);

  // sort files in alphabetical order
  const files = folder.getFiles();
  const filesArray = [];

  while (files.hasNext()) {
    filesArray.push(files.next());
  }

  // sort the array by file name
  filesArray.sort((a, b) => a.getName().localeCompare(b.getName()));

  // populate spreadsheet with images
  let i = 2;
  for (const file of filesArray) {
    const name = file.getName();
    const url = 'https://drive.google.com/uc?export=view&id=' + file.getId();
    // overwrite the relevant cells of the sheet
    sheet.getRange(i,1,1,3).setValues([[name, url, `=IMAGE("${url}", 4, 100, 100)`]]);
    console.log("appended to sheet");
    i++;
  }
}


SyntaxError: invalid syntax (1610025251.py, line 1)

### Now you should be all set to edit the sheet and add the rest of the coding info!